# 🧠 DiabetAI - Final XGBoost Training v7

**Thesis Project: Clinical Insulin Dosage Prediction**

- **Train:** Balanced dataset (50/50 action/zero)
- **Test:** Real-world imbalance (98% zeros) - The True Exam

In [ ]:
!pip install xgboost polars shap matplotlib scikit-learn -U -q

import polars as pl
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import shap
import json
import os
from sklearn.metrics import mean_squared_error, mean_absolute_error

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"
    MODEL_PATH = "/content/drive/MyDrive/DiabetesKDS/models"
except:
    BASE_PATH = r"D:\Diabete\data"
    MODEL_PATH = r"D:\Diabete\models"

os.makedirs(MODEL_PATH, exist_ok=True)
print(f"✅ XGBoost: {xgb.__version__}")
print(f"📁 Data: {BASE_PATH}")
print(f"📁 Models: {MODEL_PATH}")

## Step 1: Load Data & Define Features

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
TRAIN_FILE = f"{BASE_PATH}/processed/train_ready_balanced_v7.parquet"
TEST_FILE = f"{BASE_PATH}/processed/test_ready_real_world_v7.parquet"

# Feature definition (exact order for model)
FEATURES = [
    'CGM', 'carbs', 'iob', 'est_tdd', 'est_isf', 'est_icr',
    'cgm_delta_5min', 'cgm_delta_15min', 'cgm_velocity', 'basal_delta_5min',
    'basal', 'time_sin', 'time_cos', 'is_weekend',
    'age', 'weight', 'gender_encoded'
]

TARGET = 'bolus'

# Metadata to drop (not features)
DROP_COLS = ['id', 'date', 'gender', 'insulin_type_bolus']

print(f"🎯 Features: {len(FEATURES)}")
print(FEATURES)

In [ ]:
print("="*70)
print("📂 LOADING DATA")
print("="*70)

# Load training data (balanced)
df_train = pl.read_parquet(TRAIN_FILE)
print(f"\n✅ TRAIN: {len(df_train):,} rows")
print(f"   Action: {df_train.filter(pl.col(TARGET) > 0).height:,}")
print(f"   Zeros:  {df_train.filter(pl.col(TARGET) == 0).height:,}")

# Load test data (real-world)
df_test = pl.read_parquet(TEST_FILE)
print(f"\n✅ TEST: {len(df_test):,} rows")
action_test = df_test.filter(pl.col(TARGET) > 0).height
zero_test = df_test.filter(pl.col(TARGET) == 0).height
print(f"   Action: {action_test:,} ({action_test/len(df_test)*100:.1f}%)")
print(f"   Zeros:  {zero_test:,} ({zero_test/len(df_test)*100:.1f}%)")
print(f"   Ratio:  1:{zero_test/max(1,action_test):.0f}")

In [ ]:
# Prepare feature matrices
print("\n🔧 Preparing feature matrices...")

# Check which features are available
available_features = [f for f in FEATURES if f in df_train.columns]
missing = [f for f in FEATURES if f not in df_train.columns]

if missing:
    print(f"⚠️ Missing features: {missing}")
    FEATURES = available_features

print(f"✅ Using {len(FEATURES)} features")

# Extract features and target
X_train = df_train.select(FEATURES).to_numpy().astype(np.float32)
y_train = df_train.select(TARGET).to_numpy().ravel().astype(np.float32)

X_test = df_test.select(FEATURES).to_numpy().astype(np.float32)
y_test = df_test.select(TARGET).to_numpy().ravel().astype(np.float32)

print(f"\n📊 Train: X={X_train.shape}, y={y_train.shape}")
print(f"📊 Test:  X={X_test.shape}, y={y_test.shape}")

# Check for infinities
print(f"\n🔍 Checking for inf values...")
print(f"   Train inf: {np.isinf(X_train).sum()}")
print(f"   Test inf:  {np.isinf(X_test).sum()}")

## Step 2: Train XGBoost Model

In [ ]:
print("="*70)
print("🧠 TRAINING XGBOOST MODEL")
print("="*70)

# Check for GPU
try:
    import subprocess
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if result.returncode == 0:
        device = 'cuda'
        print("\n🚀 GPU detected - using CUDA")
    else:
        device = 'cpu'
except:
    device = 'cpu'
    print("\n💻 Using CPU")

# Model parameters
params = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'device': device,
    'random_state': 42,
    'n_jobs': -1,
    'early_stopping_rounds': 50
}

print(f"\n📋 Parameters:")
for k, v in params.items():
    print(f"   {k}: {v}")

In [ ]:
# Train model
print("\n🚀 Training...")

model = xgb.XGBRegressor(**params)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)

print(f"\n✅ Training complete!")
print(f"   Best iteration: {model.best_iteration}")
print(f"   Best score: {model.best_score:.6f}")

## Step 3: Evaluation (Real-World Exam)

In [ ]:
print("="*70)
print("📊 EVALUATION ON REAL-WORLD TEST SET")
print("="*70)

# Predict
y_pred = model.predict(X_test)

# Sanity check: clip micro-doses to 0
print("\n🔧 Applying threshold (pred < 0.1 → 0)...")
y_pred_clean = np.where(y_pred < 0.1, 0, y_pred)
y_pred_clean = np.clip(y_pred_clean, 0, 25)  # Max dose safety

# Metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred_clean))
mae = mean_absolute_error(y_test, y_pred_clean)

print(f"\n📈 METRICS (Real-World Test):")
print(f"   RMSE: {rmse:.4f} Units")
print(f"   MAE:  {mae:.4f} Units")

In [ ]:
# Clinical accuracy: predictions within ±1 unit of actual
within_1_unit = np.mean(np.abs(y_test - y_pred_clean) <= 1) * 100
within_2_units = np.mean(np.abs(y_test - y_pred_clean) <= 2) * 100

print(f"\n🎯 Clinical Accuracy:")
print(f"   Within ±1 Unit: {within_1_unit:.2f}%")
print(f"   Within ±2 Units: {within_2_units:.2f}%")

# Zero prediction accuracy
true_zeros = y_test == 0
pred_zeros = y_pred_clean == 0
zero_accuracy = np.mean(pred_zeros[true_zeros]) * 100 if true_zeros.sum() > 0 else 0

print(f"\n⚪ Zero Prediction Accuracy: {zero_accuracy:.2f}%")
print(f"   (When actual=0, model predicts 0)")

In [ ]:
# Plot 1: Scatter - Actual vs Predicted
print("\n📊 Plotting Actual vs Predicted...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full scatter
ax1 = axes[0]
ax1.scatter(y_test, y_pred_clean, alpha=0.1, s=1)
ax1.plot([0, 20], [0, 20], 'r--', lw=2, label='Perfect')
ax1.set_xlabel('Actual Bolus (Units)')
ax1.set_ylabel('Predicted Bolus (Units)')
ax1.set_title(f'Actual vs Predicted (RMSE: {rmse:.3f})')
ax1.legend()
ax1.set_xlim(0, 15)
ax1.set_ylim(0, 15)

# Error distribution
ax2 = axes[1]
errors = y_pred_clean - y_test
ax2.hist(errors, bins=100, alpha=0.7, edgecolor='black')
ax2.axvline(0, color='r', linestyle='--', lw=2)
ax2.set_xlabel('Prediction Error (Units)')
ax2.set_ylabel('Frequency')
ax2.set_title(f'Error Distribution (MAE: {mae:.3f})')
ax2.set_xlim(-5, 5)

plt.tight_layout()
plt.savefig(f"{MODEL_PATH}/evaluation_scatter_v7.png", dpi=150)
plt.show()

In [ ]:
# Plot 2: Patient Timeline (Pick a random patient with action)
print("\n📈 Plotting Patient Timeline...")

# Find patients with bolus events
df_test_eval = df_test.with_columns(pl.Series('pred_bolus', y_pred_clean))
patients_with_action = df_test_eval.filter(pl.col('bolus') > 0).select('id').unique().to_series().to_list()

if len(patients_with_action) > 0:
    # Pick first patient with action
    sample_patient = patients_with_action[0]
    df_patient = df_test_eval.filter(pl.col('id') == sample_patient).sort('date').head(500)
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # CGM
    ax1 = axes[0]
    ax1.plot(df_patient['CGM'].to_numpy(), 'b-', lw=1.5, label='CGM')
    ax1.axhline(70, color='r', linestyle='--', alpha=0.5, label='Hypo')
    ax1.axhline(180, color='orange', linestyle='--', alpha=0.5, label='High')
    ax1.set_ylabel('CGM (mg/dL)')
    ax1.set_title(f'Patient Timeline: {sample_patient}')
    ax1.legend(loc='upper right')
    ax1.set_ylim(40, 350)
    
    # Bolus
    ax2 = axes[1]
    x = np.arange(len(df_patient))
    ax2.bar(x, df_patient['bolus'].to_numpy(), alpha=0.7, label='Actual Bolus', width=1)
    ax2.plot(x, df_patient['pred_bolus'].to_numpy(), 'r-', lw=2, label='Predicted')
    ax2.set_xlabel('Time (5-min intervals)')
    ax2.set_ylabel('Bolus (Units)')
    ax2.legend(loc='upper right')
    
    plt.tight_layout()
    plt.savefig(f"{MODEL_PATH}/patient_timeline_v7.png", dpi=150)
    plt.show()
else:
    print("⚠️ No patients with bolus events found")

## Step 4: SHAP Explainability

In [ ]:
print("="*70)
print("🔍 SHAP FEATURE IMPORTANCE")
print("="*70)

# Sample for SHAP (large datasets are slow)
sample_size = min(10000, len(X_test))
indices = np.random.choice(len(X_test), sample_size, replace=False)
X_sample = X_test[indices]

print(f"\n📊 Calculating SHAP values for {sample_size:,} samples...")

# Create explainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

print("✅ SHAP values calculated!")

In [ ]:
# SHAP Summary Plot
print("\n📊 Generating SHAP Summary Plot...")

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURES, show=False)
plt.title('SHAP Feature Importance - DiabetAI v7')
plt.tight_layout()
plt.savefig(f"{MODEL_PATH}/shap_summary_v7.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance bar chart
print("\n📊 Feature Importance (Mean |SHAP|)...")

mean_shap = np.abs(shap_values).mean(axis=0)
importance_df = pl.DataFrame({
    'feature': FEATURES,
    'importance': mean_shap
}).sort('importance', descending=True)

print(importance_df)

## Step 5: Export Model

In [ ]:
print("="*70)
print("💾 EXPORTING MODEL & FEATURES")
print("="*70)

# Save model
model_file = f"{MODEL_PATH}/xgboost_clinical_v7.json"
model.save_model(model_file)
print(f"\n✅ Model saved: {model_file}")

# Save feature list
features_file = f"{MODEL_PATH}/features_v7.json"
with open(features_file, 'w') as f:
    json.dump({'features': FEATURES, 'target': TARGET}, f, indent=2)
print(f"✅ Features saved: {features_file}")

# Summary
print("\n" + "="*70)
print("🎉 TRAINING COMPLETE!")
print("="*70)
print(f"\n📊 Final Metrics:")
print(f"   RMSE: {rmse:.4f} Units")
print(f"   MAE:  {mae:.4f} Units")
print(f"   Within ±1 Unit: {within_1_unit:.2f}%")
print(f"\n📁 Artifacts:")
print(f"   Model: {model_file}")
print(f"   Features: {features_file}")
print(f"   Plots: {MODEL_PATH}/")